\begin{center}
{\LARGE \textbf{EN3160 Image Processing and Machine Vision}}\\[0.5em]
{\Large Assignment 2 -- Fitting and Alignment}\\[1.5em]

\textbf{Name:} S.A.D. Priyadarshana\\[0.3em]
\textbf{Index:} 230502X\\[0.3em]
\textbf{GitHub:} \href{https://github.com/denethp/CV-Assignment-02}{denethp/CV-Assignment-02}
\end{center}

\vspace{1em}

$\underline{\textbf{Question 1}}$

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import maximum_filter

%matplotlib inline

In [ ]:
def log_kernel(sigma):
    # Scale-normalized LoG, window of +-3 sigma;
    # mean removed so the truncated kernel gives zero response on flat regions.
    hw = int(np.ceil(3 * sigma))
    X, Y = np.meshgrid(np.arange(-hw, hw + 1), np.arange(-hw, hw + 1))
    log = 1 / (2 * np.pi * sigma ** 2) * (X ** 2 / sigma ** 2 + Y ** 2 / sigma ** 2 - 2) \
          * np.exp(-(X ** 2 + Y ** 2) / (2 * sigma ** 2))
    return log - log.mean()

In [ ]:
im = cv.imread('images/the_berry_farms_sunflower_field.jpeg', cv.IMREAD_REDUCED_COLOR_4)
gray = cv.cvtColor(im, cv.COLOR_BGR2GRAY).astype(np.float32) / 255.0

sigmas = np.arange(1, 16.5, 0.5)
scale_space = np.stack([cv.filter2D(gray, -1, log_kernel(s)) for s in sigmas], axis=2)

# Scale-space extrema: maximum in the 3x3x3 (x, y, sigma) neighbourhood above a threshold.
# Dark sunflower centres on bright petals give positive LoG peaks.
threshold = 0.2
peaks = (scale_space == maximum_filter(scale_space, size=3)) & (scale_space > threshold)
peaks[:, :, [0, -1]] = False  # end scales have only one neighbouring scale

ys, xs, ks = np.nonzero(peaks)
radii = np.sqrt(2) * sigmas[ks]  # r = sqrt(2) * sigma
print(f'sigma range: {sigmas[0]} to {sigmas[-1]} (step 0.5), blobs detected: {len(ys)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.3, 4.6))
axes[0].imshow(cv.cvtColor(im, cv.COLOR_BGR2RGB))
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(cv.cvtColor(im, cv.COLOR_BGR2RGB))
for x, y, r in zip(xs, ys, radii):
    axes[1].add_patch(plt.Circle((x, y), r, color='r', fill=False, linewidth=1))
axes[1].set_title('Detected circles (LoG)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
order = np.argsort(-radii)[:5]
print('  x    y   sigma  radius')
for o in order:
    print(f'{xs[o]:3d}  {ys[o]:3d}  {sigmas[ks[o]]:5.1f}  {radii[o]:6.2f}')

$\textbf{Discussion:}$ the image is converted to grayscale and filtered with scale-normalized LoG kernels for $\sigma \in [1, 16]$ (step 0.5). A blob is a point that is a maximum among its 26 neighbours in $(x, y, \sigma)$ and whose response exceeds 0.2; the circle radius is $r = \sqrt{2}\sigma$. The largest circles are at (338, 165) with $\sigma = 13$ ($r \approx 18.4$) and (293, 168) with $\sigma = 12$ ($r \approx 17.0$), but these lie on the dark tree line against the bright sky rather than on flowers. The largest circles on sunflowers have $\sigma = 7.5$ ($r \approx 10.6$), e.g. the large flower at (282, 338). Almost every sunflower centre is detected, and smaller $\sigma$ values pick up the more distant flowers, showing the scale selection at work. Because LoG responds to any dark blob on a lighter background, some false detections appear along the tree line and in leaf shadows; a higher threshold or restricting detection to the field region would remove these.

$\underline{\textbf{Question 2}}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Noisy point set conforming to a line and a circle
np.random.seed(0)
N = 100
half_n = N // 2
r = 10
x0_gt, y0_gt = 2, 3  # Center
s = r / 16
t = np.random.uniform(0, 2 * np.pi, half_n)
n = s * np.random.randn(half_n)
x, y = x0_gt + (r + n) * np.cos(t), y0_gt + (r + n) * np.sin(t)
X_circ = np.hstack((x.reshape(half_n, 1), y.reshape(half_n, 1)))
s = 1.
m, b = -1, 2
x = np.linspace(-12, 12, half_n)
y = m * x + b + s * np.random.randn(half_n)
X_line = np.hstack((x.reshape(half_n, 1), y.reshape(half_n, 1)))
X = np.vstack((X_circ, X_line))  # All points

$\textbf{(a) Estimating the line with RANSAC}$

Line model: $ax + by = d$ with $\|[a, b]^\top\| = 1$. Error is the normal distance $|ax_i + by_i - d|$.

In [ ]:
def line_from_points(p1, p2):
    # Unit normal [a, b] and distance d of the line through two points
    dx, dy = p2 - p1
    norm = np.hypot(dx, dy)
    a, b = dy / norm, -dx / norm
    return np.array([a, b, a * p1[0] + b * p1[1]])

def line_tls(params, pts):
    # Total least squares: sum of squared normal distances
    a, b, d = params
    return np.sum((a * pts[:, 0] + b * pts[:, 1] - d) ** 2)

# Constraint ||[a, b]|| = 1
cons = ({'type': 'eq', 'fun': lambda p: p[0] ** 2 + p[1] ** 2 - 1})

def line_consensus(params, pts, t):
    a, b, d = params
    return np.abs(a * pts[:, 0] + b * pts[:, 1] - d) < t

def ransac_line(pts, t, min_inliers, max_iterations):
    best_count = 0
    for _ in range(max_iterations):
        idx = np.random.choice(len(pts), 2, replace=False)  # minimal sample of 2 points
        model = line_from_points(pts[idx[0]], pts[idx[1]])
        inliers = line_consensus(model, pts, t)
        if inliers.sum() >= min_inliers and inliers.sum() > best_count:
            best_count = inliers.sum()
            best_sample, best_sample_model, best_inliers = pts[idx], model, inliers
    # Re-estimate the line from all inliers of the best sample
    res = minimize(line_tls, best_sample_model, args=(pts[best_inliers],), constraints=cons)
    return res.x, best_sample_model, best_sample, best_inliers

In [ ]:
t_line = 1.0  # normal-distance threshold
d_line = 40   # minimum consensus size
line_model, line_sample_model, line_sample, line_inliers = ransac_line(X, t_line, d_line, 100)

a, b, d = line_model
print(f'a = {a:.3f}, b = {b:.3f}, d = {d:.3f}, ||[a, b]|| = {np.hypot(a, b):.3f}')
print(f'slope = {-a / b:.3f}, intercept = {d / b:.3f}, inliers = {line_inliers.sum()}')

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X[:, 0], X[:, 1], s=10, color='lightgray', label='All points')
ax.scatter(X[line_inliers, 0], X[line_inliers, 1], s=10, color='b', label='Line inliers')
ax.scatter(line_sample[:, 0], line_sample[:, 1], s=40, color='r', label='Best sample for line')
x_ = np.array([-14, 14])
ax.plot(x_, (d - a * x_) / b, color='k', label='RANSAC line')
ax.plot(x_, m * x_ + 2, 'm--', label='Ground truth line')
ax.set_aspect('equal')
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

$\textbf{Parameter choice:}$ the line noise has $\sigma = 1$ in $y$, so the normal-distance noise is about $1/\sqrt{2} \approx 0.71$; a threshold $t = 1.0$ keeps most true line points. The line has 50 points, so a consensus of at least 40 is required. Each sample uses the minimum of 2 points, and the final line is re-estimated from all inliers by minimizing the total least squares error with the constraint $\|[a, b]^\top\| = 1$.